# Briefing notebook: Connected Communities

Reproduces the 3-slide briefing for **Connected Communities** (Number of people that benefit from improved access to sustainable transport infrastructure and services, `CSC_RES_ACCS_TRANS_INFRA_SERV`, FY26 snapshot) from the public World Bank APIs, and checks every slide against the numbers in the deck (✓).

| Slide | What is rebuilt |
|---|---|
| 1 · Where the portfolio stands | achieved, expected, still to deliver, delivery rate; split by World Bank, IFC, MIGA |
| 2 · Where results remain | achieved vs expected by region, ranked by still to deliver |
| 3 · Three-year priorities | WBG borrowing countries ranked by still to deliver (expected − achieved); the top 3 and the selection chart |

**Rules:** expected = what the active portfolio is expected to deliver (not a target). A delivery rate is shown only when 0 ≤ achieved ≤ expected; rows with achieved above expected are data checks and are not ranked.  
**Generated by:** Scorecard Explorer, 2026-10-17.

**Requirements:** Python 3.10+ with `pip install requests pandas plotly`. The notebook downloads everything it needs from public World Bank APIs; no key or login is required.

In [ ]:
import os, requests, pandas as pd, concurrent.futures as cf

BASE = "https://ext.api.worldbank.org/api/Scorecard/"
SNAPSHOT_FY = 2026

def scorecard_api_key():
    """The Scorecard website ships a public API key in its JavaScript bundle; read it from there
    (or set SCORECARD_API_KEY to override)."""
    if os.getenv("SCORECARD_API_KEY"):
        return os.environ["SCORECARD_API_KEY"]
    html = requests.get("https://scorecard.worldbank.org/", timeout=30).text
    bundle = next(p.split('"', 1)[0] for p in html.split('src="')[1:] if p.startswith("main."))
    js = requests.get("https://scorecard.worldbank.org/" + bundle, timeout=60).text
    return js.split('apiKey:{data:"', 1)[1].split('"', 1)[0]

HEADERS = {"apikey": scorecard_api_key(), "User-Agent": "scorecard-explorer-notebook"}

def fetch_summary(area_type, area_code, org="ALL", fy=SNAPSHOT_FY):
    """getSCSUMMARYDATA returns a Databricks SQL result set: manifest.schema.columns + result.data_array (all text)."""
    url = f"{BASE}getSCSUMMARYDATA?refareatype={area_type}&refareacode={area_code}&orgCode={org}&snapshotfy={fy}"
    for attempt in range(3):
        try:
            p = requests.get(url, headers=HEADERS, timeout=90).json()
            cols = [c["name"] for c in p["manifest"]["schema"]["columns"]]
            return pd.DataFrame(p["result"].get("data_array") or [], columns=cols)
        except Exception:
            if attempt == 2:
                raise

def totals(df):
    return (df.sub_ind_code.isna() & (df.sex_disaggregate_code == "_T") & (df.age_disaggregate_code == "_T")
            & (df.disability_disaggregate_code == "_T"))

INDICATOR = "CSC_RES_ACCS_TRANS_INFRA_SERV"
REGIONS = ["AFE", "AFW", "EAP", "ECA", "LCR", "MENAAP", "SAR"]
COUNTRIES = {"DZ": ["DZA", "MENAAP"], "MU": ["MUS", "AFE"], "AR": ["ARG", "LCR"], "VN": ["VNM", "EAP"], "BI": ["BDI", "AFE"], "NE": ["NER", "AFW"], "GH": ["GHA", "AFW"], "AO": ["AGO", "AFE"], "AG": ["ATG", "LCR"], "MN": ["MNG", "EAP"], "TG": ["TGO", "AFW"], "SR": ["SUR", "LCR"], "BJ": ["BEN", "AFW"], "IN": ["IND", "SAR"], "MR": ["MRT", "AFW"], "UA": ["UKR", "ECA"], "CF": ["CAF", "AFW"], "JO": ["JOR", "MENAAP"], "AM": ["ARM", "ECA"], "PH": ["PHL", "EAP"], "PA": ["PAN", "LCR"], "ME": ["MNE", "ECA"], "BZ": ["BLZ", "LCR"], "UZ": ["UZB", "ECA"], "SL": ["SLE", "AFW"], "KH": ["KHM", "EAP"], "CM": ["CMR", "AFW"], "KN": ["KNA", "LCR"], "KE": ["KEN", "AFE"], "BO": ["BOL", "LCR"], "BF": ["BFA", "AFW"], "ZW": ["ZWE", "AFE"], "HT": ["HTI", "LCR"], "CR": ["CRI", "LCR"], "JM": ["JAM", "LCR"], "GA": ["GAB", "AFW"], "NA": ["NAM", "AFE"], "BW": ["BWA", "AFE"], "GM": ["GMB", "AFW"], "LS": ["LSO", "AFE"], "GQ": ["GNQ", "AFW"], "GW": ["GNB", "AFW"], "SB": ["SLB", "EAP"], "MV": ["MDV", "SAR"], "ER": ["ERI", "AFE"], "ST": ["STP", "AFE"], "KM": ["COM", "AFE"], "MH": ["MHL", "EAP"], "CG": ["COG", "AFW"], "FM": ["FSM", "EAP"], "TV": ["TUV", "EAP"], "TP": ["TLS", "EAP"], "TD": ["TCD", "AFW"], "BR": ["BRA", "LCR"], "GN": ["GIN", "AFW"], "PL": ["POL", "ECA"], "LB": ["LBN", "MENAAP"], "MA": ["MAR", "MENAAP"], "ET": ["ETH", "AFE"], "EG": ["EGY", "MENAAP"], "PK": ["PAK", "MENAAP"], "EC": ["ECU", "LCR"], "WS": ["WSM", "EAP"], "TR": ["TUR", "ECA"], "BD": ["BGD", "SAR"], "RY": ["YEM", "MENAAP"], "MZ": ["MOZ", "AFE"], "BB": ["BRB", "LCR"], "LA": ["LAO", "EAP"], "MY": ["MYS", "EAP"], "LY": ["LBY", "MENAAP"], "LC": ["LCA", "LCR"], "TT": ["TTO", "LCR"], "ID": ["IDN", "EAP"], "CV": ["CPV", "AFW"], "IQ": ["IRQ", "MENAAP"], "PY": ["PRY", "LCR"], "DO": ["DOM", "LCR"], "DJ": ["DJI", "MENAAP"], "BG": ["BGR", "ECA"], "NR": ["NRU", "EAP"], "GY": ["GUY", "LCR"], "GT": ["GTM", "LCR"], "GE": ["GEO", "ECA"], "RW": ["RWA", "AFE"], "SV": ["SLV", "LCR"], "NI": ["NIC", "LCR"], "VC": ["VCT", "LCR"], "KG": ["KGZ", "ECA"], "TH": ["THA", "EAP"], "ZM": ["ZMB", "AFE"], "SC": ["SYC", "AFE"], "MD": ["MDA", "ECA"], "SN": ["SEN", "AFW"], "BY": ["BLR", "ECA"], "DM": ["DMA", "LCR"], "VU": ["VUT", "EAP"], "PG": ["PNG", "EAP"], "RU": ["RUS", "ECA"], "UG": ["UGA", "AFE"], "CL": ["CHL", "LCR"], "NG": ["NGA", "AFW"], "VE": ["VEN", "LCR"], "IR": ["IRN", "MENAAP"], "XK": ["XKX", "ECA"], "NP": ["NPL", "SAR"], "MX": ["MEX", "LCR"], "GD": ["GRD", "LCR"], "KZ": ["KAZ", "ECA"], "AL": ["ALB", "ECA"], "SY": ["SYR", "MENAAP"], "MW": ["MWI", "AFE"], "TN": ["TUN", "MENAAP"], "FJ": ["FJI", "EAP"], "ML": ["MLI", "AFW"], "CO": ["COL", "LCR"], "RS": ["SRB", "ECA"], "RO": ["ROU", "ECA"], "TJ": ["TJK", "ECA"], "LK": ["LKA", "SAR"], "HR": ["HRV", "ECA"], "ZR": ["COD", "AFE"], "AZ": ["AZE", "ECA"], "SS": ["SSD", "AFE"], "MM": ["MMR", "EAP"], "CN": ["CHN", "EAP"], "TO": ["TON", "EAP"], "TZ": ["TZA", "AFE"], "LR": ["LBR", "AFW"], "PE": ["PER", "LCR"], "HN": ["HND", "LCR"], "SO": ["SOM", "AFE"], "PW": ["PLW", "EAP"], "UY": ["URY", "LCR"], "TM": ["TKM", "ECA"], "SD": ["SDN", "AFE"], "AF": ["AFG", "MENAAP"], "BA": ["BIH", "ECA"], "SZ": ["SWZ", "AFE"], "CI": ["CIV", "AFW"], "MG": ["MDG", "AFE"], "MK": ["MKD", "ECA"], "ZA": ["ZAF", "AFE"], "BT": ["BTN", "SAR"], "KI": ["KIR", "EAP"]}  # WBG borrowers: Bank ISO2-like code -> [ISO3, region]
SHOWN = {'world': {'WBG': [222422343.0, 631688860.0], 'WB': [150919664.0, 536132858.0], 'IFC': [68179839.0, 82627727.0], 'MIGA': [3322840.0, 12928275.0]}, 'regions': {'SAR': [55607375.0, 172182178.0], 'AFE': [24190460.0, 97213141.0], 'LCR': [30940479.0, 98039907.0], 'AFW': [26582179.0, 81201161.0], 'ECA': [18163753.0, 63021118.0], 'MENAAP': [18135561.0, 57236639.0], 'EAP': [48802536.0, 62794715.0]}, 'region_people': {'SAR': 61547859.99515527, 'AFE': 326970139.91551375, 'LCR': 30078233.052007932, 'AFW': 190701238.20980185, 'ECA': None, 'MENAAP': 48257920.224116705, 'EAP': None}, 'top3': ['BGD', 'TUR', 'BRA']}  # the numbers in the deck, for the ✓ checks (None = not reported)

def tidy(raw, org):
    d = raw[totals(raw) & (raw.ind_code == INDICATOR) & (raw.org_code == org)].copy()
    d["achieved"] = pd.to_numeric(d.ind_base_achieved_val, errors="coerce")
    d["expected"] = pd.to_numeric(d.ind_base_expected_val, errors="coerce")
    # the source occasionally repeats a key; keep the row with the largest expected value
    return d.sort_values("expected", ascending=False).drop_duplicates("ref_area_code").set_index("ref_area_code")[["achieved", "expected"]]

def same(a, b, tol=0.5):
    return (pd.isna(a) and b is None) or (not pd.isna(a) and b is not None and abs(a - b) <= tol)

def rate_ok(a, e):  # a delivery rate is meaningful only when 0 <= achieved <= expected
    return pd.notna(a) and pd.notna(e) and e > 0 and 0 <= a <= e

## Data: pull everything the slides use

In [ ]:
with cf.ThreadPoolExecutor(10) as ex:
    world_raw = dict(zip(["ALL", "WB", "IFC", "MIGA"], ex.map(lambda o: fetch_summary("REGION", "ACW", o), ["ALL", "WB", "IFC", "MIGA"])))
    region_raw = pd.concat(ex.map(lambda r: fetch_summary("REGION", r), REGIONS), ignore_index=True)
    country_raw = pd.concat(ex.map(lambda c: fetch_summary("COUNTRY", c), COUNTRIES), ignore_index=True)
world = {"WBG": tidy(world_raw["ALL"], "WBG").loc["ACW"], **{o: tidy(world_raw[o], o).loc["ACW"] for o in ["WB", "IFC", "MIGA"]}}
regions = tidy(region_raw, "WBG").reindex(REGIONS)
ctry = pd.DataFrame.from_dict(COUNTRIES, orient="index", columns=["iso3", "region"]).join(tidy(country_raw, "WBG"))
print(len(ctry), "borrowing countries;", int(ctry.expected.notna().sum()), "with results")

Context used on slide 2: **people without reliable year-round transport** = population × (100 − share) of `SP_ROD_R2KM` (Scorecard client context), population from Data360 for the same year. It does not select countries.

In [ ]:
ctx = country_raw[totals(country_raw) & (country_raw.ind_code == "SP_ROD_R2KM")].copy()
ctx["value"] = pd.to_numeric(ctx.ind_base_achieved_val, errors="coerce").fillna(pd.to_numeric(ctx.ind_achieved_val, errors="coerce"))
ctx["year"] = ctx.ind_time_per.map(lambda s: [int(x) for x in str(s).split("-") if len(x) == 4 and x.isdigit()][-1])  # end year of a period or range
ctx = ctx.dropna(subset=["value"]).sort_values("year").groupby("ref_area_code").last()[["value", "year"]]
rows, skip = [], 0
while True:  # Data360 population, all pages (1,000 rows each)
    d = requests.get("https://data360api.worldbank.org/data360/data", params={"DATABASE_ID": "WB_WDI", "INDICATOR": "WB_WDI_SP_POP_TOTL",
        "timePeriodFrom": 2015, "timePeriodTo": 2026, "top": 1000, "skip": skip}, timeout=60).json()
    rows += d.get("value") or []
    if len(d.get("value") or []) < 1000: break
    skip += 1000
pop = pd.DataFrame(rows).assign(year=lambda x: x.TIME_PERIOD.astype(int), pop=lambda x: x.OBS_VALUE.astype(float))
def pop_for(iso3, year):  # same year as the need value, else the latest earlier year
    p = pop[(pop.REF_AREA == iso3) & ((pop.year <= year) if pd.notna(year) else True)]
    return p.sort_values("year")["pop"].iloc[-1] if len(p) else None
ctry = ctry.join(ctx.rename(columns={"value": "need", "year": "need_year"}))
ctry["people"] = [pop_for(i, y) for i, y in zip(ctry.iso3, ctry.need_year)]
ctry["people"] = ctry.people * (100 - ctry.need) / 100
print(int(ctry.people.notna().sum()), "countries with a people-in-need value")

## Slide 1 · Where the portfolio stands

In [ ]:
w = world["WBG"]
slide1 = pd.Series({"achieved": w.achieved, "expected": w.expected, "still to deliver": w.expected - w.achieved,
                   "delivery rate": w.achieved / w.expected if rate_ok(w.achieved, w.expected) else None})
inst = pd.DataFrame({o: world[o] for o in ["WB", "IFC", "MIGA"]}).T
for o in ["WBG", "WB", "IFC", "MIGA"]:
    a, e = SHOWN["world"][o] or (None, None)
    print("✓" if same(world[o].achieved, a) and same(world[o].expected, e) else "⚠ (updated since snapshot?)", o, world[o].achieved, world[o].expected)
display(slide1.to_frame("all countries"), inst)

## Slide 2 · Where results remain to be delivered (regions)

In [ ]:
regions["still_to_deliver"] = regions.expected - regions.achieved
regions["delivery_rate"] = [a / e if rate_ok(a, e) else None for a, e in zip(regions.achieved, regions.expected)]
regions["data_check"] = regions.achieved > regions.expected
regions["people_in_need"] = ctry.groupby("region").people.sum(min_count=1).reindex(REGIONS)
regions = regions.sort_values("still_to_deliver", ascending=False)
for r, row in regions.iterrows():
    a, e = SHOWN["regions"].get(r, (None, None))
    p = SHOWN["region_people"].get(r)
    ok_p = same(row.people_in_need, p, tol=max(1, 1e-6 * (p or 0)))
    print("✓" if same(row.achieved, a) and same(row.expected, e) and ok_p else "⚠ (updated since snapshot?)", r)
import plotly.graph_objects as go
t = regions.iloc[::-1]
labels = ["above expected (data check)" if c else f"{r:.0%}" for r, c in zip(t.delivery_rate.fillna(0), t.data_check)]
fig = go.Figure([go.Bar(y=t.index, x=t.expected, orientation="h", name="Expected", marker_color="#cde7f9"),
                 go.Bar(y=t.index, x=t.achieved, orientation="h", name="Achieved", marker_color="#0071bc", width=0.4,
                        text=labels, textposition="outside")])
fig.update_layout(barmode="overlay", template="simple_white", height=420, title="Achieved vs expected by region")
fig.show()
regions

## Slide 3 · Three-year priorities (one rule: most still to deliver)

In [ ]:
clean = ctry[ctry.achieved.notna() & (ctry.expected > 0) & (ctry.achieved <= ctry.expected)].copy()  # data checks are not ranked
clean["still_to_deliver"] = clean.expected - clean.achieved
clean["delivered_share"] = clean.achieved / clean.expected
ranked = clean[clean.still_to_deliver > 0].sort_values("still_to_deliver", ascending=False)
top3 = ranked.head(3)
print("✓" if list(top3.iso3) == SHOWN["top3"] else "⚠ (updated since snapshot?)", "top 3:", list(top3.iso3))
g = ranked.still_to_deliver.values
cut = (g[2] * g[3]) ** 0.5 if len(g) > 3 else g[-1] * 0.85  # line between the 3rd and 4th country
sel = ranked.index.isin(top3.index)
fig = go.Figure([go.Scatter(x=100 * ranked.delivered_share[~sel], y=ranked.still_to_deliver[~sel], mode="markers", name="Other countries",
                            marker=dict(color="#c3c9cf", size=7), text=ranked.iso3[~sel]),
                 go.Scatter(x=100 * top3.delivered_share, y=top3.still_to_deliver, mode="markers+text", name="Most still to deliver",
                            marker=dict(color="#eb6834", size=13), text=top3.iso3, textposition="top center")])
fig.add_hline(y=cut, line_dash="dash", annotation_text="Top 3 still to deliver ↑")
fig.update_layout(template="simple_white", height=460, yaxis_type="log", xaxis_title="Share of expected results delivered so far (%)",
                  yaxis_title="Results still to deliver", title="Why these countries")
fig.show()
print("not ranked (achieved above expected):", list(ctry[ctry.achieved > ctry.expected].iso3))
top3

In [ ]:
ranked.to_csv(f"briefing_{INDICATOR}_FY{SNAPSHOT_FY}_countries.csv")
regions.to_csv(f"briefing_{INDICATOR}_FY{SNAPSHOT_FY}_regions.csv")
print("saved", len(ranked), "countries and", len(regions), "regions")